 # TUTORIAL: POD and SPOD classes



 Demonstrates the `POD` and `SPOD` classes from `tools.pod_spod`,

 applied to DNS data of the wake past a circular cylinder at Re = 100.



 **Contents**

 * [1 · Setup and data loading](#1-setup)

 * [2 · Fit POD](#2-fit)

 * [3 · Energy spectrum](#3-spectrum)

 * [4 · Spatial modes](#4-modes)

 * [5 · Temporal coefficients](#5-temporal)

 * [6 · Encode, decode and reconstruct](#6-reconstruct)

 * [7 · SPOD extension (Sieber 2016)](#7-spod)



 **Key classes and functions**

 ```python

 from tools.pod_spod import POD, SPOD          # sklearn-style ROM classes

 from tools.pod_spod import prepare_data       # NaN masking + zero-mean

 ```

 ---

 ## 1 · Setup and data loading  <a class="anchor" id="1-setup"></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils import set_working_directories, get_wake_data

from tools import POD, SPOD
from tools import prepare_data, energy_fraction

# ── locate data ──────────────────────────────────────────────────────────────
data_folder = set_working_directories('wakes')[0]
case        = 'circle_re_100'

get_wake_data(data_folder, case=case)   # downloads from Zenodo if not present
print(f'Data folder: {data_folder}')


In [ ]:
from scipy.io import loadmat

mat    = loadmat(f'{data_folder}/{case}.mat')
ux_raw = mat['ux']   # (N_t, Nx, Ny)  —  NaN at cylinder body
uy_raw = mat['uy']
pp_raw = mat['pp']
N_t, Nx, Ny = ux_raw.shape

print(f'Snapshots : N_t = {N_t}')
print(f'Grid      : {Nx} × {Ny}')

# prepare_data:
#   • detects NaN mask (cylinder body) from the first snapshot
#   • flattens to (N_fluid, N_t) and subtracts temporal mean
#   • returns to_grid() callable for re-embedding modes on the 2-D mesh
#
# Stack ux + uy so modes capture both components simultaneously.
Q, fluid_mask, to_grid = prepare_data([ux_raw, uy_raw], subtract_mean=True)
N_fluid = fluid_mask.sum()

print(f'\nData matrix Q : {Q.shape}   (2 × {N_fluid} fluid pts, {N_t} snapshots)')
print(f'Cylinder body : {(~fluid_mask).sum()} NaN pts excluded')


In [ ]:
# ── Quick look at the flow field ──────────────────────────────────────────────
snap = np.where(~fluid_mask, np.nan, ux_raw[N_t // 2] - np.nanmean(ux_raw, axis=0))

fig, ax = plt.subplots(figsize=(11, 3.5))
vmax = np.nanpercentile(np.abs(snap), 99)
im   = ax.pcolormesh(snap.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label="$u'_x$")
ax.set_aspect('equal'); ax.set_xlabel('x index'); ax.set_ylabel('y index')
ax.set_title(f'Streamwise velocity fluctuation  —  snapshot {N_t//2} / {N_t}')
plt.tight_layout(); plt.show()


 ---

 ## 2 · Fit POD  <a class="anchor" id="2-fit"></a>



 The `POD` class wraps snapshot POD (Sirovich 1987) with a clean sklearn-style API:



 ```python

 pod = POD(n_modes, method)   # method: 'exact' | 'randomized' (default)

 pod.fit(Q)                   # learn modes from zero-mean Q (N_x, N_t)

 ```



 After `fit`, the following attributes are available:



 | Attribute | Shape | Description |

 |---|---|---|

 | `pod.Psi` | $(N_x, N_{\rm modes})$ | Spatial modes (orthonormal columns) |

 | `pod.Phi` | $(N_{\rm modes}, N_t)$ | Temporal coefficients |

 | `pod.Sigma` | $(N_{\rm modes},)$ | Singular values $\Sigma_k = \sqrt{\lambda_k}$ |

 | `pod.Q_mean` | $(N_x, 1)$ | Temporal mean (stored for encode/decode) |

In [ ]:
N_modes = 20

pod = POD(n_modes=N_modes, method='exact').fit(Q)

print(f'Fitted POD: {pod.N_modes} modes')
print(f'Psi   shape: {pod.Psi.shape}   (N_fluid * 2,  N_modes)')
print(f'Phi   shape: {pod.Phi.shape}   (N_modes, N_t)')
print(f'Sigma shape: {pod.Sigma.shape}')
print(f'\nLeading singular values: {pod.Sigma[:6].round(4)}')


 ---

 ## 3 · Energy spectrum  <a class="anchor" id="3-spectrum"></a>



 The energy captured by mode $k$ is $\lambda_k = \Sigma_k^2$.

 The **relative energy fraction** is $\lambda_k / \sum_j \lambda_j$.



 Use the built-in `POD.plot_spectrum(pod)`, or `energy_fraction(pod.Sigma)` for the raw numbers.

In [ ]:
rel, cum = pod.energy_fraction()
print(f'Relative energy — mode 1: {rel[0]*100:.2f}%,  modes 1–2: {cum[1]*100:.2f}%,  modes 1–4: {cum[3]*100:.2f}%')

# Built-in plot: bar chart + cumulative energy
POD.plot_spectrum(pod, max_mode=N_modes)
plt.suptitle('POD energy spectrum — cylinder wake Re = 100  [ux + uy]', fontsize=12, y=1.01)
plt.show()

print('\n► Two dominant modes (~paired eigenvalues) = Kármán vortex shedding conjugate pair.')
print('  Modes 3–4 capture the first harmonic.')


 ---

 ## 4 · Spatial modes  <a class="anchor" id="4-modes"></a>



 Each column of `pod.Psi` is a flat vector over fluid points.

 Split by field and restore to the 2-D grid with `to_grid`.

In [ ]:
n_show = 4

# Psi rows: first N_fluid = ux component, next N_fluid = uy component
Psi_ux = pod.Psi[:N_fluid, :]    # (N_fluid, N_modes)
Psi_uy = pod.Psi[N_fluid:, :]    # (N_fluid, N_modes)

fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(2, n_show, hspace=0.45, wspace=0.25)

for k in range(n_show):
    for row, Psi_c, label in [(0, Psi_ux, '$u_x$'), (1, Psi_uy, '$u_y$')]:
        mode_2d = to_grid(Psi_c[:, k])
        vmax    = np.nanpercentile(np.abs(mode_2d), 98)
        ax = fig.add_subplot(gs[row, k])
        im = ax.pcolormesh(mode_2d.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                           shading='auto')
        plt.colorbar(im, ax=ax, shrink=0.75)
        energy_pct = rel[k] * 100
        ax.set_title(f'Mode {k+1}  ({energy_pct:.1f}%)  {label}', fontsize=9)
        ax.set_aspect('equal')
        if row == 1:
            ax.set_xlabel('x index')
        ax.set_ylabel('y index')

fig.suptitle('POD spatial modes — cylinder wake Re = 100', fontsize=12)
plt.show()


 ---

 ## 5 · Temporal coefficients  <a class="anchor" id="5-temporal"></a>



 `pod.Phi[k, :]` is the amplitude of mode $k$ at each snapshot.

 For a periodic flow, the leading pair $(\phi_1, \phi_2)$ should trace a **circle** in phase space.

In [ ]:
# Built-in imshow of the full coefficient matrix
POD.plot_time_coefficients(pod, num_modes=10)
plt.suptitle('Temporal coefficient matrix $\\Phi$ (first 10 modes)', fontsize=11, y=1.01)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Time series of modes 1 & 2
axes[0].plot(pod.Phi[0, :], lw=0.8, label='$\\phi_1$')
axes[0].plot(pod.Phi[1, :], lw=0.8, ls='--', label='$\\phi_2$')
axes[0].set_xlabel('Snapshot index'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Temporal coefficients — modes 1 & 2'); axes[0].legend()

# Phase portrait
axes[1].plot(pod.Phi[0, :], pod.Phi[1, :], lw=0.5, color='steelblue')
axes[1].set_xlabel('$\\phi_1$'); axes[1].set_ylabel('$\\phi_2$')
axes[1].set_title('Phase portrait $(\\phi_1, \\phi_2)$ — limit cycle')
axes[1].set_aspect('equal')

# Frequency content of first 4 modes
N_fft = N_t
freqs = np.fft.rfftfreq(N_fft)
for k in range(4):
    fft_k = np.abs(np.fft.rfft(pod.Phi[k, :]))
    axes[2].semilogy(freqs, fft_k, lw=1.2, alpha=0.85, label=f'mode {k+1}')
axes[2].set_xlim(0, 0.35)
axes[2].set_xlabel('Normalised frequency'); axes[2].set_ylabel('|FFT|')
axes[2].set_title('Spectral content of $\\phi_k(t)$'); axes[2].legend(fontsize=8)

plt.suptitle('POD temporal coefficients — cylinder wake Re = 100', fontsize=11)
plt.tight_layout(); plt.show()

print('► Circular phase portrait = travelling-wave conjugate pair at the shedding frequency.')


 ---

 ## 6 · Encode, decode and reconstruct  <a class="anchor" id="6-reconstruct"></a>



 ```python

 Z     = pod.encode(Q)          # project Q onto modes  →  (N_modes, N_t)

 Q_hat = pod.decode(Z)          # lift back to state space  →  (N_x, N_t)

 Q_hat = pod.reconstruct(Q)     # round-trip  encode → decode

 mse   = pod.score(Q)           # mean squared reconstruction error

 ```



 Reconstruction with $r < N_{\rm modes}$ modes is done via `reconstruct(Q, n_modes=r)`.

In [ ]:
# ── encode / decode ───────────────────────────────────────────────────────────
Z     = pod.encode(Q)          # (N_modes, N_t)
Q_hat = pod.decode(Z)          # (N_fluid*2, N_t)

print(f'Latent Z    : {Z.shape}')
print(f'Q_hat shape : {Q_hat.shape}')
print(f'encode(Q) == Phi? ', np.allclose(Z, pod.Phi, atol=1e-10))

# ── score ─────────────────────────────────────────────────────────────────────
mse_full = pod.score(Q)
print(f'\nReconstruction MSE ({N_modes} modes): {mse_full:.2e}')


In [ ]:
# ── Reconstruction error vs number of modes ───────────────────────────────────
mode_range = [1, 2, 4, 6, 8, 10, 15, 20]
mse_list   = [pod.score(pod.reconstruct(Q, n_modes=r)) for r in mode_range]




In [ ]:
fig = plt.figure(figsize=(12, 4), layout='constrained')
gs  = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1, 2, 2], wspace=0.1)

# left: MSE vs modes
ax_mse = fig.add_subplot(gs[0])
ax_mse.semilogy(mode_range, mse_list, 'o-', color='steelblue')
ax_mse.set_xlabel('Number of modes $r$'); ax_mse.set_ylabel('Reconstruction MSE')

# right: three flow-field panels
rec_modes = [mode_range[1], mode_range[-1]]  

snap_idx = N_t // 2

for gsi, rm in zip(range(1, 3), rec_modes):
    Q_r2     = pod.reconstruct(Q, n_modes=rm)
    vmax     = np.nanpercentile(np.abs(to_grid(Q[:N_fluid, snap_idx])), 98)

    gs_right = gs[gsi].subgridspec(3, 1)

    for col, (data, title, cmap) in enumerate([
            (to_grid(Q[:N_fluid, snap_idx]),                                    'Original $u_x$', 'RdBu_r'),
            (to_grid(Q_r2[:N_fluid, snap_idx]),                                 f'{rm}-mode recon.',  'RdBu_r'),
            (to_grid(np.abs(Q[:N_fluid, snap_idx] - Q_r2[:N_fluid, snap_idx])), 'Absolute error', 'Reds'),
        ]):
        ax = fig.add_subplot(gs_right[col])

        kw = dict(vmin=-vmax, vmax=vmax) if cmap == 'RdBu_r' else dict(vmin=0, vmax=vmax)
        im = ax.pcolormesh(data.T, cmap=cmap, shading='auto', **kw)
        plt.colorbar(im, ax=ax, shrink=0.8, pad=0.04)
        ax.set_title(title, fontsize=9); 
        ax.set_aspect('equal')
        ax.set_ylabel('y')
        if col == 2:
            ax.set_xlabel('x')
        else:
            ax.set_xticklabels([])
    

fig.suptitle(f'Reconstruction quality  —  snapshot {snap_idx}', fontsize=12)
plt.show()


 ---

 ## 7 · SPOD extension (Sieber 2016)  <a class="anchor" id="7-spod"></a>



 The `SPOD` class has the **same API** as `POD` with one extra parameter:



 ```python

 spod = SPOD(Nf=Nf, filter_kind='gaussian').fit(Q)

 ```



 Internally, it replaces the temporal correlation matrix $\mathbf{C}$ by its filtered version

 $\widetilde{\mathbf{C}} = \mathbf{G}^\top \mathbf{C}\,\mathbf{G}$ before the eigendecomposition.

 Setting `Nf=0` exactly recovers standard snapshot POD.



 Here we choose `Nf` equal to half the estimated shedding period — the natural scale for

 a filter that should resolve the vortex shedding without over-smoothing.

In [ ]:
# ── Estimate shedding period from leading temporal coefficient ─────────────────
fft_phi1 = np.abs(np.fft.rfft(pod.Phi[0, :]))
f_peak   = np.fft.rfftfreq(N_t)[np.argmax(fft_phi1)]
T_snaps  = int(round(1.0 / f_peak))            # snapshots per shedding cycle
Nf       = T_snaps // 2

print(f'Dominant normalised frequency : f = {f_peak:.4f}')
print(f'Estimated shedding period     : ~{T_snaps} snapshots')
print(f'Filter half-width Nf          :  {Nf}')

# ── Fit SPOD ──────────────────────────────────────────────────────────────────
spod = SPOD(Nf=Nf, filter_kind='gaussian', n_modes=N_modes).fit(Q)
print(f'\nSPOD fitted: {spod.N_modes} modes')


In [ ]:
# ── POD vs SPOD: leading mode spatial comparison ──────────────────────────────
# Layout: n_comp rows (one per mode), 2 columns (POD left, SPOD right)
n_comp = 10
fig = plt.figure(figsize=(12, 2 * n_comp), layout='constrained')
gs  = gridspec.GridSpec(n_comp, 2)

for k in range(n_comp):
    for col, (psi_src, label) in enumerate([
            (pod.Psi[:N_fluid, :],  'POD'),
            (spod.Psi[:N_fluid, :], f'SPOD $N_f$={Nf}'),
        ]):
        mode_2d = to_grid(psi_src[:, k])
        vmax    = np.nanpercentile(np.abs(mode_2d), 98)
        ax = fig.add_subplot(gs[k, col])
        im = ax.pcolormesh(mode_2d.T, cmap='RdBu_r',
                           vmin=-vmax, vmax=vmax, shading='auto')
        fig.colorbar(im, ax=ax, shrink=0.6)
        ax.set_title(f'{label}  mode {k+1}', fontsize=9)
        ax.set_aspect('equal')
        ax.set_ylabel('y')
        if k == n_comp - 1:
            ax.set_xlabel('x')
        else:
            ax.set_xticklabels([])

plt.show()


In [ ]:
# ── Quantitative agreement ────────────────────────────────────────────────────
cos_sim = np.array([
    abs(pod.Psi[:, k] @ spod.Psi[:, k]) /
    (np.linalg.norm(pod.Psi[:, k]) * np.linalg.norm(spod.Psi[:, k]))
    for k in range(N_modes)
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, N_modes+1), cos_sim, color='steelblue', edgecolor='white')
axes[0].axhline(1.0, lw=1, ls='--', color='gray')
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel('Mode index'); axes[0].set_ylabel('|cos similarity|')
axes[0].set_title(f'Spatial agreement: POD vs SPOD ($N_f$={Nf})')

# Spectral content: POD mode 1 vs SPOD mode 1
freqs_plot = np.fft.rfftfreq(N_t)
axes[1].semilogy(freqs_plot, np.abs(np.fft.rfft(pod.Phi[0, :])),
                 lw=1.5, alpha=0.8, label='POD mode 1')
axes[1].semilogy(freqs_plot, np.abs(np.fft.rfft(spod.Phi[0, :])),
                 lw=1.5, alpha=0.8, ls='--', label=f'SPOD mode 1 ($N_f$={Nf})')
axes[1].axvline(f_peak, color='tomato', lw=1.2, ls=':', label=f'$f_s$={f_peak:.3f}')
axes[1].set_xlim(0, 4*f_peak)
axes[1].set_xlabel('Normalised frequency'); axes[1].set_ylabel('|FFT|')
axes[1].set_title('Spectral content of leading temporal coefficient')
axes[1].legend(fontsize=9)

plt.suptitle('POD vs SPOD — validation on single-frequency wake', fontsize=12)
plt.tight_layout(); plt.show()

print('\n For this single-frequency flow, SPOD reproduces POD modes with near-unity similarity.')
print('  SPOD adds the most value for multi-frequency flows (harmonics, competing instabilities).')


 ---

 ## Summary



 ### POD class API



 ```python

 from tools.pod_spod import prepare_data

 from tools.pod_spod import POD, SPOD



 Q, mask, to_grid = prepare_data(ux_raw)          # or [ux_raw, uy_raw] for stacked fields



 pod = POD(n_modes=20, method='exact').fit(Q)     # 'randomized' is faster for large data

 Z   = pod.encode(Q)                              # (N_modes, N_t) — latent representation

 Q_r = pod.decode(Z)                              # (N_x, N_t)    — state-space reconstruction

 Q_r = pod.reconstruct(Q, n_modes=4)              # round-trip with optional truncation

 mse = pod.score(Q)                               # reconstruction MSE



 POD.plot_spectrum(pod)                           # energy bar chart + cumulative

 POD.plot_time_coefficients(pod, num_modes=10)    # imshow of Phi

 mode_2d = to_grid(pod.Psi[:N_fluid, k])         # reshape mode k to (Nx, Ny)

 ```



 ### SPOD — one extra line vs POD



 ```python

 spod = SPOD(Nf=Nf, filter_kind='gaussian').fit(Q)   # everything else identical

 ```



 | | POD | SPOD |

 |---|---|---|

 | Extra parameter | — | `Nf` (filter half-width) |

 | Correlation matrix | $\mathbf{C}$ | $\widetilde{\mathbf{C}} = \mathbf{G}^\top\mathbf{C}\mathbf{G}$ |

 | Mode optimality | maximum energy | maximum energy per temporal scale |

 | Limiting case | `Nf=0` is POD | `Nf→N_t/2` → DFT |



 See `dev/dev_scripts/SPOD_Sieber2016.ipynb` for a detailed derivation and comparison with Towne SPOD.